# Warfarin QML — VQC k=4 Primary (seed 42) | Reproducibility notebook

**Mục đích:** Giữ mã chạy lại **VQC primary** cho 5-fold CV (600 mẫu Quantum Development Pool, PCA k=4, seed 42), bổ sung tính tái lập còn thiếu trong repository. Không phải thí nghiệm mới, không chọn lại mô hình.

- **Chế độ mặc định: `RUN_TRAINING=False`** — chỉ kiểm tra raw/split, 5 JSON primary đã lưu, predictions, metrics, và in bảng summary. **Không huấn luyện, không tạo output.**
- Chỉ khi thực sự cần tái lập thí nghiệm, đổi `RUN_TRAINING=True` và chọn `FOLDS_TO_RUN=(0,)` trước; dùng máy Conda `nt409`. Kết quả chạy lại sẽ lưu sang tên riêng `docs/results/vqc_k4_primary_reproduced_fold*.json`; không bao giờ ghi đè 5 JSON primary gốc.
- **Không dùng Final Test** để fit, đánh giá hoặc chọn hyperparameter. Tất cả preprocessing, PCA, scaler fit trên 480 mẫu training mỗi fold.
- Cấu hình primary: `zz_feature_map` 4 qubits, reps 2/linear; `real_amplitudes` reps 1/linear (8 parameters); `StatevectorSampler` 1.024 shots, seed 42; `COBYLA(maxiter=60)`; 3 nhóm với `interpret(bitstring)=bitstring%3`.
- Số liệu `fit_seconds` khác nhau theo máy/lần chạy. Dùng metrics/predictions lưu sẵn làm kết quả primary đã kiểm chứng; nếu chạy lại, so sánh kết quả và ghi phiên bản môi trường, không mặc định ghi đè lịch sử.

## Cell 1 — Imports và cấu hình kiểm tra (mặc định không train)

In [1]:
from pathlib import Path
from time import perf_counter
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import yaml
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score, recall_score, confusion_matrix)

from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.optimizers import COBYLA


def find_project_root(start):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "configs/feature_policy.yaml").is_file() and (path / "data/splits/quantum_dev_folds.csv").is_file():
            return path
    raise FileNotFoundError("Run notebook from inside Warfarin project")

ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.prepare_stage4 import load_and_validate_dataset, verify_raw_file
from src.preprocessing import build_preprocessor

OUT = ROOT / "docs/results"
SPLITS = ROOT / "data/splits"
CLASSES = ["Low", "Intermediate", "High"]
SEED = 42
K = 4
MAXITER = 60
SHOTS = 1024
RUN_TRAINING = False
FOLDS_TO_RUN = (0,)  # Used only when RUN_TRAINING=True

assert set(FOLDS_TO_RUN).issubset(set(range(5)))
print("[PASS] Python:", sys.executable)
print("[PASS] Project:", ROOT)
print("[PASS] VQC primary seed 42, 4 qubits, 5 locked folds")
print("[MODE]", "TRAINING (optional)" if RUN_TRAINING else "VERIFY_ONLY (no retraining)")

[PASS] Python: c:\Users\admin\anaconda3\envs\nt409\python.exe
[PASS] Project: D:\Warfarin-Quantum-Machine-Learning-Classification
[PASS] VQC primary seed 42, 4 qubits, 5 locked folds
[MODE] VERIFY_ONLY (no retraining)


## Cell 2 — Kiểm tra dữ liệu, checksum và khóa split

In [2]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

verify_raw_file()
with (ROOT / "configs/feature_policy.yaml").open(encoding="utf-8") as f:
    policy = yaml.safe_load(f)
assert policy["training_allowed"] is True
approved = list(policy["approved_X_columns"])
assert len(approved) == len(set(approved)) == 10

manifest_path = SPLITS / "split_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest["approved_features"] == approved
for filename, expected_sha in manifest["files"].items():
    assert sha256_file(SPLITS / filename) == expected_sha, filename

X, y = load_and_validate_dataset(approved)
labels = pd.read_csv(ROOT / "data/interim/labels.csv")
folds = pd.read_csv(SPLITS / "quantum_dev_folds.csv")
partitions = pd.read_csv(SPLITS / "final_split.csv")
y_array = np.asarray(y, dtype=str)  # Qiskit requires position-indexed labels
assert len(X) == len(y_array) == len(labels) == 5410
assert labels["raw_row_index"].is_unique and folds["raw_row_index"].is_unique
assert np.array_equal(y_array, labels["target_group"].astype(str).to_numpy())
assert set(folds["fold_id"].astype(int)) == set(range(5))
qdev_ids = set(folds["raw_row_index"].astype(int))
dev_ids = set(partitions.loc[partitions["partition"] == "development", "raw_row_index"].astype(int))
test_ids = set(partitions.loc[partitions["partition"] == "final_test", "raw_row_index"].astype(int))
assert len(qdev_ids) == 600 and len(dev_ids) == 4328 and len(test_ids) == 1082
assert qdev_ids.issubset(dev_ids) and qdev_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)
assert dev_ids | test_ids == set(labels["raw_row_index"].astype(int))
for f in range(5):
    assert (folds["fold_id"] == f).sum() == 120

print("[PASS] Raw checksum, feature policy and split manifest")
print("[PASS] 600 qdev samples, 5 folds (480 train / 120 val)")
print("[PASS] All 1082 Final Test samples isolated")

[PASS] Raw checksum, feature policy and split manifest
[PASS] 600 qdev samples, 5 folds (480 train / 120 val)
[PASS] All 1082 Final Test samples isolated


## Cell 3 — Chỉ đọc và kiểm định 5 kết quả primary đã lưu

Chỉ số được **tính lại từ predictions** và đối chiếu CSV/JSON, không lấy nguyên số từ màn hình Terminal. Cell này không fit mô hình.

In [3]:
archive_metrics = pd.read_csv(OUT / "vqc_k4_5fold_metrics.csv")
archive_predictions = pd.read_csv(OUT / "vqc_k4_5fold_predictions.csv")
assert len(archive_metrics) == 5 and len(archive_predictions) == 600
assert set(archive_metrics["fold"].astype(int)) == set(range(5))
rows = []
all_ids = set()
all_cms = np.zeros((3, 3), dtype=int)
initial_point_reference = None
for f in range(5):
    saved = json.loads((OUT / f"vqc_k4_fold{f}_diagnostic.json").read_text(encoding="utf-8"))
    for key, value in {"fold": f, "train_n": 480, "val_n": 120, "pca_k": K, "seed": SEED,
                       "maxiter": MAXITER, "shots": SHOTS, "optimizer": "COBYLA",
                       "feature_map": "zz_feature_map", "ansatz": "real_amplitudes",
                       "multiclass_mapping": "bitstring_modulo_3", "final_test_used": False}.items():
        assert saved[key] == value, (f, key)
    assert saved["class_order"] == CLASSES
    assert len(saved["loss_history"]) == MAXITER
    assert saved["objective_evaluations"] == MAXITER
    assert np.isfinite(saved["loss_history"]).all()
    init = np.asarray(saved["initial_point"], dtype=float)
    assert len(init) == 8 and np.isfinite(init).all()
    if initial_point_reference is None:
        initial_point_reference = init
    else:
        assert np.allclose(init, initial_point_reference)

    df = pd.DataFrame(saved["predictions"])
    ids = set(df["raw_row_index"].astype(int))
    expected = set(folds.loc[folds["fold_id"] == f, "raw_row_index"].astype(int))
    assert len(df) == 120 and ids == expected and ids.isdisjoint(test_ids)
    assert ids.isdisjoint(all_ids)
    all_ids |= ids
    label_map = dict(zip(labels["raw_row_index"].astype(int), labels["target_group"].astype(str)))
    assert all(a == label_map[int(r)] for r, a in zip(df["raw_row_index"], df["actual"]))
    assert set(df["predicted"]).issubset(set(CLASSES))

    from_csv = archive_predictions.loc[archive_predictions["fold"] == f]
    assert len(from_csv) == 120
    expected_pred = from_csv.set_index("raw_row_index")[["actual", "predicted"]].sort_index()
    saved_pred = df.set_index("raw_row_index")[["actual", "predicted"]].sort_index()
    pd.testing.assert_frame_equal(saved_pred, expected_pred, check_dtype=False)

    a = df["actual"].to_numpy()
    p = df["predicted"].to_numpy()
    cm = confusion_matrix(a, p, labels=CLASSES)
    all_cms += cm
    calc = {
        "fold": f,
        "macro_f1": f1_score(a, p, labels=CLASSES, average="macro", zero_division=0),
        "accuracy": accuracy_score(a, p),
        "balanced_accuracy": balanced_accuracy_score(a, p),
        "recall_high": recall_score(a, p, labels=CLASSES, average=None, zero_division=0)[2],
        "fit_seconds": float(saved["fit_seconds"]),
        "predict_seconds": float(saved["predict_seconds"]),
    }
    assert np.array_equal(cm, np.asarray(saved["confusion_matrix"], dtype=int))
    original_csv = archive_metrics.loc[archive_metrics["fold"] == f].iloc[0]
    for metric in ("macro_f1", "accuracy", "balanced_accuracy", "recall_high", "fit_seconds", "predict_seconds"):
        assert np.isclose(float(calc[metric]), float(saved[metric]) if metric in saved else float(original_csv[metric]))
        assert np.isclose(float(calc[metric]), float(original_csv[metric]))
    rows.append(calc)
    print(f"[PASS] Primary seed={SEED}, Fold {f}, Macro-F1={calc['macro_f1']:.4f}")

assert all_ids == qdev_ids and len(all_ids) == 600
baseline_report = pd.DataFrame(rows)
print("\n=== VQC PRIMARY 5-FOLD SUMMARY ===")
print(baseline_report.drop(columns="fold").agg(["mean", "std"]).round(4))
print("\n=== AGGREGATE CONFUSION MATRIX ===")
print(pd.DataFrame(all_cms, index=CLASSES, columns=CLASSES))
print("[PASS] Saved primary results verified without retraining")

[PASS] Primary seed=42, Fold 0, Macro-F1=0.3936
[PASS] Primary seed=42, Fold 1, Macro-F1=0.3457
[PASS] Primary seed=42, Fold 2, Macro-F1=0.3505
[PASS] Primary seed=42, Fold 3, Macro-F1=0.3166
[PASS] Primary seed=42, Fold 4, Macro-F1=0.3088

=== VQC PRIMARY 5-FOLD SUMMARY ===
      macro_f1  accuracy  balanced_accuracy  recall_high  fit_seconds  \
mean    0.3431    0.3833             0.3713       0.3250     161.4014   
std     0.0335    0.0680             0.0170       0.2162      92.6450   

      predict_seconds  
mean           0.8159  
std            0.6416  

=== AGGREGATE CONFUSION MATRIX ===
              Low  Intermediate  High
Low            81            67    50
Intermediate  105           124    97
High           25            26    25
[PASS] Saved primary results verified without retraining


## Cell 4 — Hàm tái lập huấn luyện VQC (chỉ được gọi nếu bật `RUN_TRAINING`)

Lưu kết quả chạy lại dưới tên mới `vqc_k4_primary_reproduced_foldN.json`. Bản archived primary không bị sửa. Cấu hình model giống 5 JSON gốc, và **validation chỉ dùng sau khi đã fit xong**.

In [4]:
def reproduce_primary_fold(fold_id):
    fold_id = int(fold_id)
    assert fold_id in range(5)
    output = OUT / f"vqc_k4_primary_reproduced_fold{fold_id}.json"
    if output.exists():
        raise FileExistsError(f"Cannot overwrite reproduced output: {output}")

    val_ids = set(folds.loc[folds["fold_id"] == fold_id, "raw_row_index"].astype(int))
    train_ids = qdev_ids - val_ids
    assert len(train_ids) == 480 and len(val_ids) == 120
    assert train_ids.isdisjoint(val_ids) and (train_ids | val_ids).isdisjoint(test_ids)
    train_pos = np.flatnonzero(labels["raw_row_index"].isin(train_ids).to_numpy())
    val_pos = np.flatnonzero(labels["raw_row_index"].isin(val_ids).to_numpy())
    X_train, X_val = X.iloc[train_pos], X.iloc[val_pos]
    y_train, y_val = y_array[train_pos], y_array[val_pos]
    assert len(y_train) == 480 and len(y_val) == 120

    prep = build_preprocessor(approved)
    encoded_train = prep.fit_transform(X_train)
    encoded_val = prep.transform(X_val)
    pca = PCA(n_components=4, svd_solver="full")
    pca_train = pca.fit_transform(encoded_train)
    pca_val = pca.transform(encoded_val)
    scaler = MinMaxScaler(feature_range=(0.0, float(np.pi)))
    z_train = np.clip(scaler.fit_transform(pca_train), 0.0, np.pi)
    z_val = np.clip(scaler.transform(pca_val), 0.0, np.pi)
    assert z_train.shape == (480, 4) and z_val.shape == (120, 4)
    assert np.isfinite(z_train).all() and np.isfinite(z_val).all()

    feature_map = zz_feature_map(feature_dimension=4, reps=2, entanglement="linear")
    ansatz = real_amplitudes(num_qubits=4, reps=1, entanglement="linear")
    assert ansatz.num_parameters == 8
    init = np.random.default_rng(SEED).uniform(-np.pi, np.pi, size=ansatz.num_parameters)
    assert np.allclose(init, initial_point_reference)
    history = []

    def callback(weights, objective):
        history.append(float(objective))
        if len(history) % 10 == 0:
            print(f"Fold {fold_id}: eval={len(history)}, loss={objective:.6f}", flush=True)

    model = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=MAXITER),
        sampler=StatevectorSampler(default_shots=SHOTS, seed=SEED),
        loss="cross_entropy",
        interpret=lambda bitstring: bitstring % 3,
        output_shape=3,
        initial_point=init,
        callback=callback,
    )
    start = perf_counter()
    model.fit(z_train, y_train)
    fit_seconds = perf_counter() - start
    start = perf_counter()
    pred = np.asarray(model.predict(z_val)).reshape(-1)
    predict_seconds = perf_counter() - start
    assert len(pred) == 120 and set(pred).issubset(set(CLASSES))
    assert len(history) == MAXITER and np.isfinite(history).all()

    cm = confusion_matrix(y_val, pred, labels=CLASSES)
    recalls = recall_score(y_val, pred, labels=CLASSES, average=None, zero_division=0)
    result = {
        "experiment": "VQC_k4_primary_reproduced", "status": "REPRODUCED_NOT_PRIMARY_ARCHIVE",
        "fold": fold_id, "train_n": 480, "val_n": 120, "pca_k": 4,
        "feature_map": "zz_feature_map", "feature_map_reps": 2,
        "feature_map_entanglement": "linear", "ansatz": "real_amplitudes",
        "ansatz_reps": 1, "ansatz_parameters": 8,
        "multiclass_mapping": "bitstring_modulo_3", "optimizer": "COBYLA",
        "maxiter": MAXITER, "shots": SHOTS, "seed": SEED,
        "initial_point": init.tolist(), "objective_evaluations": len(history),
        "loss_history": history,
        "macro_f1": float(f1_score(y_val, pred, labels=CLASSES, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_val, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_val, pred)),
        "recall_high": float(recalls[2]),
        "confusion_matrix": cm.tolist(), "class_order": CLASSES,
        "predictions": [
            {"raw_row_index": int(labels.iloc[pos]["raw_row_index"]),
             "actual": str(actual), "predicted": str(predicted)}
            for pos, actual, predicted in zip(val_pos, y_val, pred)
        ],
        "fit_seconds": float(fit_seconds), "predict_seconds": float(predict_seconds),
        "feature_policy_sha256": sha256_file(ROOT / "configs/feature_policy.yaml"),
        "split_manifest_sha256": sha256_file(manifest_path),
        "final_test_used": False,
    }
    with output.open("x", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2, allow_nan=False)
    print(f"[PASS] Reproduced VQC Fold {fold_id}: Macro-F1={result['macro_f1']:.4f}")
    print("[SAVED]", output)
    return result

## Cell 5 — Optional: chạy lại một hoặc nhiều fold (**mặc định bỏ qua**)

**Không thay đổi `RUN_TRAINING=False` chỉ để đóng gói hoặc bàn giao.** Kết quả 5-fold đã có đầy đủ trong archive. Nếu nhóm muốn kiểm tra khả năng chạy lại thực tế, bật `RUN_TRAINING=True`, trước tiên chạy **chỉ Fold 0**, và đánh giá chênh lệch với kết quả cũ. Mỗi lần chạy có thể mất vài phút tùy máy.

In [5]:
if not RUN_TRAINING:
    print("[SKIP] Re-training disabled. Archived primary VQC results preserved.")
else:
    for fold_id in FOLDS_TO_RUN:
        reproduced = reproduce_primary_fold(fold_id)
        baseline = json.loads((OUT / f"vqc_k4_fold{fold_id}_diagnostic.json").read_text(encoding="utf-8"))
        print(f"Fold {fold_id} Macro-F1: baseline={baseline['macro_f1']:.6f}, "
              f"reproduced={reproduced['macro_f1']:.6f}; no overwrite")

[SKIP] Re-training disabled. Archived primary VQC results preserved.


## Checklist cuối

- `[PASS]` nguồn, 4 split checksum, 600 qdev, 5 folds và Final Test isolated.
- `[PASS]` cả 5 JSON seed=42 được kiểm tra, metrics tính lại từ predictions và khớp CSV primary đã lưu.
- **Giữ**: notebook này, `scripts/prepare_stage4.py`, `src/preprocessing.py`, các JSON/CSV primary.
- **Không xóa** file gốc hoặc mở Final Test vì mục đích kiểm tra tái lập.
- **VQC robustness** ở notebook khác thuộc exploratory; không thay thế seed 42.
- Đây là dữ liệu nghiên cứu hồi cứu, không sử dụng để ra quyết định liều thuốc thực tế.